<a href="https://colab.research.google.com/github/Deepanshu911/CoalIntel-AI-/blob/main/sih.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP 1 - run once on a NEW Colab machine.
# First set the runtime: Runtime > Change runtime type > T4 GPU
!apt -q install -y tesseract-ocr > /dev/null
!pip -q install -U pillow "pandas==2.2.3" pymupdf pdfplumber openpyxl python-docx reportlab sentence-transformers faiss-cpu transformers accelerate pytesseract wordcloud gradio

# Restart the runtime so the new packages load cleanly.
# Colab will say "session crashed" - that is expected. Then run STEP 2.
import os
os.kill(os.getpid(), 9)




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 5.5 MB/s eta 0:00:00


In [ ]:
# ============================================================
# SIH 2026 | PS 26023 | AI-powered reporting prototype (all-in-one cell)
# Run AFTER the install cell (which restarts the runtime).
# Needs: sih26023/data/ on Google Drive with the ch02, ch08, ch09, ch12, ch14 PDFs.
# ============================================================
RUN_TESTS = False   # set True once to print accuracy tests before the app starts

# %% setup
from google.colab import drive
drive.mount('/content/drive')
import os, re, io, glob, warnings
import numpy as np, pandas as pd
import pymupdf, pdfplumber, pytesseract
from PIL import Image
from docx import Document
warnings.filterwarnings("ignore")

BASE = '/content/drive/MyDrive/sih26023'
os.makedirs(BASE + '/data', exist_ok=True)
os.makedirs(BASE + '/out', exist_ok=True)
DUMMY = {"production_2022_23.pdf", "production_2023_24.xlsx"}

# %% documents
def chunk_text(text, size=800, overlap=150):
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i + size])
        i += size - overlap
    return out

def page_text(page, dpi=200):
    text = page.get_text().strip()
    if len(text) > 30:
        return text, False
    pix = page.get_pixmap(dpi=dpi)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    return pytesseract.image_to_string(img).strip(), True

def build_chunks(folder):
    out = []
    for f in sorted(glob.glob(folder + '/*.pdf')):
        if os.path.basename(f) in DUMMY:
            continue
        doc = pymupdf.open(f)
        for pno, page in enumerate(doc, 1):
            text, ocr = page_text(page)
            if not text:
                continue
            for c in chunk_text(text):
                out.append({
                    "text": c,
                    "file": os.path.basename(f),
                    "page": pno,
                    "ocr": ocr
                })
    return out

chunks = build_chunks(BASE + '/data')
assert chunks, "No PDFs found in sih26023/data on Google Drive"
print(len(chunks), "chunks,", sum(c["ocr"] for c in chunks), "from OCR")

# %% search
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer('all-MiniLM-L6-v2')

emb = embedder.encode(
    [c["text"] for c in chunks],
    normalize_embeddings=True,
    show_progress_bar=False
)

index = faiss.IndexFlatIP(emb.shape[1])
index.add(np.array(emb, dtype='float32'))

def retrieve(q, k=4):
    qv = embedder.encode([q], normalize_embeddings=True)
    scores, ids = index.search(
        np.array(qv, dtype='float32'),
        k
    )
    return [
        (chunks[i], float(s))
        for i, s in zip(ids[0], scores[0])
    ]

# %% model
from transformers import pipeline
import torch

_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(
    "Language model device:",
    "GPU" if torch.cuda.is_available()
    else "CPU (slow: switch runtime to T4 GPU)"
)

llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=_dtype,
    device_map="auto"
)

def _numbers(text):
    nums = [
        n.replace(",", "")
        for n in re.findall(r"\d[\d,]*\.?\d*", text)
    ]

    return [
        n for n in nums
        if len(n.replace(".", "")) >= 3
        and not re.fullmatch(r"(19|20)\d\d", n)
    ]

def rag_answer(q, k=6, show=2):
    """Retrieval + small language model."""
    hits = retrieve(q, k)

    context = "\n\n".join(
        f"[{c['file']} p.{c['page']}]\n{c['text']}"
        for c, _ in hits
    )

    msgs = [
        {
            "role": "system",
            "content":
                "Answer ONLY from the context, in one short sentence "
                "giving the figure and the year. "
                "Do not add any other claims or comments. "
                "If the answer is not in the context, say "
                "'Not found in the documents.'"
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {q}"
        }
    ]

    try:
        result = llm(
            msgs,
            max_new_tokens=120,
            do_sample=False
        )

        generated = result[0]["generated_text"]

        if isinstance(generated, list):
            out = generated[-1]["content"]
        else:
            out = str(generated)

        out = out.strip()

    except Exception as e:
        print("RAG ERROR:", repr(e))
        return f"Error while answering the question: {str(e)}", []

    if "Not found" in out:
        return out, []

    nums = _numbers(out)

    cited = [
        (c, s)
        for c, s in hits
        if any(
            n in c["text"].replace(",", "")
            for n in nums
        )
    ]

    pool = sorted(
        cited if cited else hits,
        key=lambda h: -h[1]
    )[:show]

    return out, [
        f"{c['file']} p.{c['page']} (score {s:.2f})"
        for c, s in pool
    ]

def verified(out, q, k=6):
    """Every number in the answer must appear in retrieved text."""
    ctx = " ".join(
        c["text"]
        for c, _ in retrieve(q, k)
    ).replace(",", "")

    return all(
        n in ctx
        for n in _numbers(out)
    )

# %% module1
def extract_company_table(f, name):
    """Template rule for company-wise production table."""
    recs = []
    num = re.compile(r"\d[\d,]*\.?\d*")

    with pdfplumber.open(f) as pdf:
        for pno, page in enumerate(pdf.pages, 1):

            for t in page.extract_tables():

                for r in t:
                    if len(r) < 4 or not r[0]:
                        continue

                    label = str(r[0]).replace("*", "").strip()

                    if (
                        not label
                        or "total" in label.lower()
                        or label.lower() == "company"
                    ):
                        continue

                    nums = num.findall(
                        str(r[3] or "")
                    )

                    if (
                        len(nums) != 4
                        or not str(r[2] or "").strip()
                    ):
                        continue

                    for fy, val in [
                        ("2019-20", r[2]),
                        ("2020-21", nums[1]),
                        ("2021-22", nums[3])
                    ]:

                        recs.append({
                            "Subsidiary": label,
                            "FY": fy,
                            "Production (MT)": str(val)
                                .replace(",", "")
                                .strip(),
                            "source_file": name,
                            "source_page": pno
                        })

    return recs

def text_rows(f, name, labels=("ECL", "BCCL")):
    """ECL and BCCL sit as plain text lines above the table."""
    recs = []

    doc = pymupdf.open(f)

    for pno, page in enumerate(doc, 1):

        lines = [
            l.strip()
            for l in page.get_text().splitlines()
            if l.strip()
        ]

        for i, l in enumerate(lines):

            if l.replace("*", "") in labels:

                vals = []

                for nxt in lines[i + 1:i + 14]:

                    if re.search(r"[A-Za-z]{2,}", nxt):
                        break

                    vals += [
                        t for t in nxt.split()
                        if re.fullmatch(
                            r"-?\d[\d,]*\.?\d*",
                            t
                        )
                    ]

                    if len(vals) >= 6:
                        break

                if len(vals) >= 6:

                    for fy, k in [
                        ("2019-20", 1),
                        ("2020-21", 3),
                        ("2021-22", 5)
                    ]:

                        recs.append({
                            "Subsidiary":
                                l.replace("*", ""),
                            "FY": fy,
                            "Production (MT)":
                                vals[k].replace(",", ""),
                            "source_file": name,
                            "source_page": pno
                        })

    return recs

COLS = [
    "Subsidiary",
    "FY",
    "Production (MT)",
    "source_file",
    "source_page"
]

def extract_files(paths):

    recs = []

    pat = re.compile(
        r"([A-Za-z]{2,10})\s+"
        r"(\d{4}-\d{2})\s+"
        r"(-?[\d,]+\.?\d*)"
    )

    for f in paths:

        name = os.path.basename(f)

        if f.lower().endswith(".pdf"):

            found = extract_company_table(
                f,
                name
            )

            if found:

                found += text_rows(
                    f,
                    name
                )

            else:

                doc = pymupdf.open(f)

                for pno, page in enumerate(doc, 1):

                    for m in pat.finditer(
                        page.get_text()
                    ):

                        found.append({
                            "Subsidiary": m.group(1),
                            "FY": m.group(2),
                            "Production (MT)":
                                m.group(3).replace(",", ""),
                            "source_file": name,
                            "source_page": pno
                        })

            recs += found

        elif f.lower().endswith(
            (".xlsx", ".csv")
        ):

            df = (
                pd.read_excel(f)
                if f.lower().endswith(".xlsx")
                else pd.read_csv(f)
            )

            df["source_file"] = name
            df["source_page"] = "Sheet1"

            recs += df.to_dict("records")

    if not recs:
        return pd.DataFrame(
            columns=COLS
        )

    d = pd.DataFrame(recs)

    d["Production (MT)"] = pd.to_numeric(
        d["Production (MT)"],
        errors="coerce"
    )

    return d

SUBS = [
    "ECL", "BCCL", "CCL", "NCL",
    "WCL", "SECL", "MCL", "NEC"
]

def cross_check(d):

    for fy in d["FY"].unique():

        sub = d[
            (d.FY == fy)
            & d.Subsidiary.isin(SUBS)
        ]

        cil = d[
            (d.FY == fy)
            & (d.Subsidiary == "CIL")
        ]

        if (
            len(cil) == 1
            and sub.Subsidiary.nunique() == len(SUBS)
            and sub["Production (MT)"].notna().all()
        ):

            diff = abs(
                cil["Production (MT)"].iloc[0]
                - sub["Production (MT)"].sum()
            )

            idx = cil.index[0]

            if d.loc[idx, "validation"] == "OK":

                d.loc[idx, "validation"] = (
                    "OK (equals sum of subsidiaries)"
                    if diff < 0.5
                    else
                    f"Mismatch: CIL differs from "
                    f"subsidiary sum by {diff:.2f} MT"
                )

    return d

def validate(d):

    d = d.copy()

    d["validation"] = "OK"

    for i, r in d.iterrows():

        v = r["Production (MT)"]

        if pd.isna(v):
            d.loc[i, "validation"] = "Missing value"

        elif v < 0:
            d.loc[i, "validation"] = "Negative value"

        elif v > 1000:
            d.loc[i, "validation"] = (
                "Out of plausible range (>1000 MT)"
            )

    for i in d[
        d.duplicated(
            ["Subsidiary", "FY"],
            keep="first"
        )
    ].index:

        d.loc[i, "validation"] = (
            "Duplicate Subsidiary+FY"
        )

    return cross_check(d)

def make_report(d, path):

    doc = Document()

    doc.add_heading(
        "Coal Production Report",
        0
    )

    ok = d[
        d.validation.str.startswith("OK")
    ]

    doc.add_paragraph(
        f"{len(ok)} of {len(d)} records passed validation; "
        f"{len(d) - len(ok)} flagged."
    )

    t = doc.add_table(
        rows=1,
        cols=5
    )

    for c, h in zip(
        t.rows[0].cells,
        [
            "Subsidiary",
            "FY",
            "MT",
            "Source",
            "Status"
        ]
    ):
        c.text = h

    for _, r in d.iterrows():

        c = t.add_row().cells

        c[0].text = str(r.Subsidiary)
        c[1].text = str(r.FY)

        c[2].text = (
            ""
            if pd.isna(r["Production (MT)"])
            else str(r["Production (MT)"])
        )

        c[3].text = (
            f"{r.source_file} "
            f"(p.{r.source_page})"
        )

        c[4].text = r.validation

    doc.save(path)

def run_report(files):

    if not files:
        return None, None

    d = validate(
        extract_files(files)
    )

    d = d.sort_values(
        ["source_file", "FY"],
        kind="stable"
    ).reset_index(drop=True)

    out = BASE + "/out/report_ui.docx"

    make_report(
        d,
        out
    )

    return d, out

# %% lookup
try:

    tbl = validate(
        extract_files([
            BASE + '/data/ch09_production.pdf'
        ])
    )

    print(
        "Table lookup ready:",
        len(tbl),
        "rows"
    )

except Exception as e:

    print(
        "Table lookup disabled:",
        e
    )

    tbl = pd.DataFrame(
        columns=COLS + ["validation"]
    )

ALIASES = {
    "CIL": ["coal india", "cil"],
    "SCCL": ["sccl", "singareni"],
    "Captive": ["captive"],
    "BCCL": ["bccl"],
    "ECL": ["ecl"],
    "CCL": ["ccl"],
    "NCL": ["ncl"],
    "WCL": ["wcl"],
    "SECL": ["secl"],
    "MCL": ["mcl"],
    "NEC": ["nec"]
}

def table_answer(q):

    ql = q.lower()

    m = re.search(
        r"(20\d\d)\s*-\s*(\d{2})",
        ql
    )

    if (
        not m
        or not any(
            w in ql
            for w in ["produc", "output"]
        )
    ):
        return None

    fy = f"{m.group(1)}-{m.group(2)}"

    ents = [
        e for e, al in ALIASES.items()
        if any(
            re.search(
                rf"\b{re.escape(a)}\b",
                ql
            )
            for a in al
        )
    ]

    if len(ents) != 1:
        return None

    r = tbl[
        (tbl.Subsidiary == ents[0])
        & (tbl.FY == fy)
    ]

    if r.empty:
        return None

    r = r.iloc[0]

    return (
        f"{ents[0]} produced "
        f"{round(float(r['Production (MT)']), 3)} "
        f"million tonnes in {fy}.",
        [
            f"{r.source_file} p.{r.source_page}"
        ]
    )

def answer(q, k=6, show=2):

    t = table_answer(q)

    if t:
        return t

    out, src = rag_answer(
        q,
        k,
        show
    )

    if (
        "Not found" not in out
        and not verified(out, q, k)
    ):
        return (
            "Could not verify this figure "
            "in the documents.",
            []
        )

    return out, src

# %% module2
from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer,
    ENGLISH_STOP_WORDS
)

from sklearn.decomposition import (
    LatentDirichletAllocation
)

from wordcloud import WordCloud

WC_PATH = BASE + "/out/wordcloud.png"
TOPICS_PATH = BASE + "/out/topics.csv"

def build_module2():

    extra_stop = {
        "annual", "report", "2022", "2023",
        "2021", "2022-23", "million",
        "tonnes", "tonne", "mt", "mte",
        "table", "year", "total", "state",
        "per", "cent", "ministry", "coal",
        "india", "limited", "cil", "page",
        "chapter", "actual", "target",
        "upto", "wise", "company", "sector",
        "power", "sccl", "nlcil", "given",
        "provisional", "rate"
    }

    stop = list(
        ENGLISH_STOP_WORDS
        | extra_stop
    )

    docs = [
        re.sub(
            r"\s+",
            " ",
            re.sub(
                r"[^a-z\s]",
                " ",
                c["text"].lower()
            )
        ).strip()
        for c in chunks
    ]

    tfidf = TfidfVectorizer(
        stop_words=stop,
        token_pattern=r"[a-z]{4,}",
        max_df=0.6,
        min_df=2
    )

    X = tfidf.fit_transform(docs)

    kw = pd.Series(
        X.sum(axis=0).A1,
        index=tfidf.get_feature_names_out()
    ).sort_values(
        ascending=False
    )

    WordCloud(
        width=1000,
        height=500,
        background_color="white",
        colormap="viridis"
    ).generate_from_frequencies(
        kw.head(80).to_dict()
    ).to_file(WC_PATH)

    cv = CountVectorizer(
        stop_words=stop,
        token_pattern=r"[a-z]{4,}",
        max_df=0.6,
        min_df=3
    )

    C = cv.fit_transform(docs)

    lda = LatentDirichletAllocation(
        n_components=5,
        random_state=42,
        learning_method="batch"
    ).fit(C)

    words = cv.get_feature_names_out()

    rows = [
        {
            "Topic": i + 1,
            "Top keywords":
                ", ".join(
                    words[j]
                    for j in comp.argsort()[-10:][::-1]
                )
        }
        for i, comp in enumerate(
            lda.components_
        )
    ]

    pd.DataFrame(rows).to_csv(
        TOPICS_PATH,
        index=False
    )

if not (
    os.path.exists(WC_PATH)
    and os.path.exists(TOPICS_PATH)
):

    build_module2()

    print(
        "Word cloud and topics generated"
    )

def show_topics():
    return (
        WC_PATH,
        pd.read_csv(TOPICS_PATH)
    )

# %% tests
if RUN_TESTS:

    tests = [
        (
            "How much coal did Coal India Limited produce in 2021-22?",
            ["622.63"],
            "table"
        ),
        (
            "What was the coal production of Coal India Limited in 2020-21?",
            ["596.22"],
            "table"
        ),
        (
            "What was the coal production of NCL in 2021-22?",
            ["122.43"],
            "table"
        ),
        (
            "What was the coal production of MCL in 2021-22?",
            ["168.17", "168.168"],
            "table"
        ),
        (
            "What was the coal production of SECL in 2021-22?",
            ["142.51"],
            "table"
        ),
        (
            "What was the coal production of CCL in 2021-22?",
            ["68.85", "68.846"],
            "table"
        ),
        (
            "What was the coal production of WCL in 2020-21?",
            ["50.27", "50.28"],
            "table"
        ),
        (
            "What was the coal production of ECL in 2019-20?",
            ["50.40"],
            "table"
        ),
        (
            "What was the coal production of SCCL in 2021-22?",
            ["65.02"],
            "table"
        ),
        (
            "What was the captive and commercial coal production in 2021-22?",
            ["85.71", "85.72"],
            "table"
        ),
        (
            "What was the coal production of WCL in 2021-22?",
            ["57.708"],
            "table"
        ),
        (
            "What was the coal production of SECL in 2020-21?",
            ["150.606"],
            "table"
        ),
        (
            "What was the coal production of MCL in 2019-20?",
            ["140.358"],
            "table"
        ),
        (
            "What was the coal production of BCCL in 2021-22?",
            ["30.511"],
            "table"
        ),
        (
            "What was the coal production of SCCL in 2020-21?",
            ["50.58"],
            "table"
        ),
        (
            "What was the coal production of NCL in 2019-20?",
            ["108.053"],
            "table"
        ),
        (
            "What was the actual coal supply in 2022-23 up to December 2022?",
            ["786.469", "786.47"],
            "text"
        ),
        (
            "What was the estimated overall coal demand for 2022-23?",
            ["1204"],
            "text"
        )
    ]

    refusals = [
        "What is the price of coal in Australia in 2030?",
        "How many coal mines are in Brazil?",
        "Who won the cricket world cup?"
    ]

    score = {
        "table": [0, 0],
        "text": [0, 0]
    }

    for q, exp, grp in tests:

        a, src = answer(q)

        ok = any(
            e in a
            for e in exp
        )

        score[grp][0] += ok
        score[grp][1] += 1

        print(
            ("PASS " if ok else "FAIL "),
            q,
            "->",
            a,
            "|",
            "; ".join(src) if src else "-"
        )

    for g, (ok, n) in score.items():

        print(
            f"{g} questions: "
            f"{ok} of {n} correct"
        )

    ref = [
        any(
            s in answer(q)[0]
            for s in (
                "Not found",
                "Could not verify"
            )
        )
        for q in refusals
    ]

    print(
        f"out-of-scope refused: "
        f"{sum(ref)} of {len(ref)}"
    )

# %% ui
import gradio as gr

def run_qa(q):

    try:

        if not q or not q.strip():
            return "Please enter a question."

        a, src = answer(q)

        if src:
            return (
                a
                + "\n\nSources:\n- "
                + "\n- ".join(src)
            )

        return a

    except Exception as e:

        print(
            "QA ERROR:",
            repr(e)
        )

        return f"Error: {str(e)}"

with gr.Blocks(
    title="CMPDI/CIL AI Reporting"
) as demo:

    # UPDATED HEADING
    gr.Markdown(
        "# AI-Powered Geological & Mining Reporting"
    )

    with gr.Tab(
        "1. Report Generation"
    ):

        f = gr.File(
            file_count="multiple",
            type="filepath",
            label="Upload PDF / Excel / CSV"
        )

        b1 = gr.Button(
            "Extract, validate and generate report"
        )

        tbl_out = gr.Dataframe(
            label="Extracted data with source tracing and validation"
        )

        docx_out = gr.File(
            label="Download DOCX report"
        )

        b1.click(
            run_report,
            f,
            [tbl_out, docx_out]
        )

    with gr.Tab(
        "2. Word Cloud & Topics"
    ):

        b2 = gr.Button(
            "Show word cloud and topics"
        )

        img = gr.Image(
            label="Word cloud"
        )

        tp = gr.Dataframe(
            label="Topics"
        )

        b2.click(
            show_topics,
            None,
            [img, tp]
        )

    with gr.Tab(
        "3. Ask a Question"
    ):

        q = gr.Textbox(
            label="Question",
            placeholder=
                "How much coal did Coal India produce in 2021-22?"
        )

        b3 = gr.Button("Ask")

        ans = gr.Textbox(
            label="Answer with sources",
            lines=8
        )

        b3.click(
            run_qa,
            q,
            ans
        )

demo.launch(
    share=True,
    debug=True
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
250 chunks, 0 from OCR


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Language model device: GPU


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Table lookup ready: 36 rows
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://343b90833f72523043.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
!pip install -q pymupdf pdfplumber pytesseract pillow python-docx


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 53.9 MB/s eta 0:00:00
